In [1]:
import torch
import numpy as np
from model import SASRecModel, negative_sampling_loss

In [2]:
def ndcg_at_k(rel, pred, k=10):
    ndcg = 0.0
    if rel in pred[:k]:
        pred_list = list(pred[:k])
        score = pred_list.index(rel) + 1
        ndcg = 1.0 / np.log2(score + 1)
        return ndcg
    return ndcg


def recall_at_k(rel, pred, k=10):
    return 1.0 if rel in pred[:k] else 0.0

In [3]:
# Тест 1: размерность выхода без контентных фич - модульное тестирование
print("Тест 1: Размерность выхода модели")
model = SASRecModel(cnt_item=100, max_seq_len=10)
x = torch.randint(1, 100, (4, 10))
out = model(x)
print(f"Output shape: {out.shape}")
assert out.shape == (4, 10, 101), f'Ожидалось (4, 10, 101), получили {out.shape}'
print('Тест пройден')

Тест 1: Размерность выхода модели
Output shape: torch.Size([4, 10, 101])
Тест пройден


In [4]:
# Тест 2: размерность выхода с контентными фичами — модульное тестирование
print("Тест 2: Размерность выхода с контентными фичами")
model = SASRecModel(cnt_item=100, max_seq_len=10, cnt_authors=50, cnt_categories=30)
x = torch.randint(1, 100, (4, 10))
a = torch.randint(1, 50, (4, 10))
c = torch.randint(1, 30, (4, 10))
out = model(x, a, c)
print(f"Output shape: {out.shape}")
assert out.shape == (4, 10, 101)
print('Тест пройден')

Тест 2: Размерность выхода с контентными фичами
Output shape: torch.Size([4, 10, 101])
Тест пройден


In [5]:
# Тест 3: каузальная маска — тестирование методом белого ящика
print("Тест 3: Каузальная маска")
seq_len = 5
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
print(mask)
assert mask[0, 1] == True   # позиция 0 не видит позицию 1
assert mask[1, 0] == False  # позиция 1 видит позицию 0
assert mask[1, 2] == True   # позиция 1 не видит позицию 2
print('Тест пройден')

Тест 3: Каузальная маска
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
Тест пройден


In [6]:
# Тест 4: predict_next — модульное тестирование
print("Тест 4: predict_next")
model = SASRecModel(cnt_item=100, max_seq_len=10)
history = [5, 10, 15, 20]
items, scores = model.predict_next(history, top_k=5)
print(f"Items: {items}")
assert len(items) == 5
assert len(scores) == 5
print('Тест пройден')

Тест 4: predict_next
Items: [83 10 78 65 16]
Тест пройден


In [7]:
# Тест 5: пустая история (негативный тест) — негативное тестирование
print("Тест 5: Пустая история")
model = SASRecModel(cnt_item=100, max_seq_len=10)
try:
    items, scores = model.predict_next([], top_k=10)
    assert len(items) == 10
    print(f"Items: {items}")
    print('Тест пройден (пустая история обработана)')
except Exception as e:
    print(f'Ошибка: {e}')
print()

Тест 5: Пустая история
Items: [ 37  59  24  31  20  76  75  84   9 100]
Тест пройден (пустая история обработана)



In [8]:
# Тест 6: метрики NDCG и Recall — модульное тестирование
print("Тест 6: Метрики NDCG и Recall")
assert abs(ndcg_at_k(5, [5, 1, 2, 3, 4], k=5) - 1.0) < 0.001
assert ndcg_at_k(99, [1, 2, 3, 4, 5], k=5) == 0.0
assert recall_at_k(3, [1, 2, 3, 4, 5], k=5) == 1.0
assert recall_at_k(99, [1, 2, 3, 4, 5], k=5) == 0.0
print('Тест пройден')

Тест 6: Метрики NDCG и Recall
Тест пройден


In [9]:
# Тест 7: Negative Sampling Loss — модульное тестирование
print("Тест 7: Negative Sampling Loss")
logits = torch.tensor([[0.5, 2.0, 0.1]])
targets = torch.tensor([1])
loss = negative_sampling_loss(logits, targets, cnt_item=3, num_negatives=2)
print(f"Loss: {loss.item():.4f}")
assert loss.item() > 0
print('Тест пройден')

Тест 7: Negative Sampling Loss
Loss: 2.9983
Тест пройден


In [10]:
# Тест 8: интеграционный тест с контентными фичами — интеграционное тестирование
print("Тест 8: Интеграционный тест (с контентными фичами)")
model_test = SASRecModel(cnt_item=100, cnt_authors=50, cnt_categories=30)
history = [1, 2, 3, 4, 5, 10, 8, 7, 4, 5]
items, scores = model_test.predict_next(history, top_k=10)
assert len(items) == 10
assert all(0 <= i <= 100 for i in items)
assert all(isinstance(s, (float, np.floating)) for s in scores)
print('Тест пройден')

Тест 8: Интеграционный тест (с контентными фичами)
Тест пройден


In [11]:
# Тест 9: проверка эмбеддингов авторов и категорий — модульное тестирование
print("Тест 9: Эмбеддинги авторов и категорий")
model_test = SASRecModel(cnt_item=100, cnt_authors=50, cnt_categories=30)
assert hasattr(model_test, 'author_emb')
assert hasattr(model_test, 'category_emb')
assert model_test.author_emb.num_embeddings == 51  # 50 + 1 padding
assert model_test.category_emb.num_embeddings == 31  # 30 + 1 padding
print('Тест пройден')

Тест 9: Эмбеддинги авторов и категорий
Тест пройден


In [12]:
# Тест 10: обновление весов при обучении — тестирование методом белого ящика
print("Тест 10: Обновление весов при обучении")
model_test = SASRecModel(cnt_item=100, cnt_authors=20, cnt_categories=10)
optimizer = torch.optim.Adam(model_test.parameters(), lr=0.01)
x = torch.randint(1, 100, (4, 10))
a = torch.randint(1, 20, (4, 10))
c = torch.randint(1, 10, (4, 10))
targets = torch.randint(1, 100, (4,))
weights_before = model_test.item_emb.weight.data.clone()
logits = model_test(x, a, c)
loss = negative_sampling_loss(logits[:, -1, :], targets, cnt_item=100, num_negatives=10)
optimizer.zero_grad()
loss.backward()
optimizer.step()
weights_after = model_test.item_emb.weight.data
assert not torch.equal(weights_before, weights_after)
print('Тест пройден')

Тест 10: Обновление весов при обучении
Тест пройден
